# Préparation Finale — Features & Cibles
Entrée : `features_encodees.parquet` → Sortie : `X.parquet` + `y.parquet`

- Sélection des 5 cibles `out.*`
- Transformation logarithmique (log1p) des colonnes asymétriques
- Gestion des NaN résiduels
- Séparation features / cibles

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'

df = pd.read_parquet(DATA_PROCESSED / 'features_encodees.parquet')
print('Dataset chargé :', df.shape)

Dataset chargé : (549971, 384)


## 1. Sélection des cibles

In [2]:
TARGETS = [
    'out.electricity.total.energy_consumption..kwh',
    'out.natural_gas.total.energy_consumption..kwh',
    'out.fuel_oil.total.energy_consumption..kwh',
    'out.propane.total.energy_consumption..kwh',
    'out.emissions.total.lrmer_mid_case_25..co2e_kg',
]

TARGETS = [c for c in TARGETS if c in df.columns]

out_all  = [c for c in df.columns if c.startswith('out.')]
out_drop = [c for c in out_all if c not in TARGETS]

df.drop(columns=out_drop, inplace=True)
print(f'Cibles conservées : {len(TARGETS)}')

Cibles conservées : 5


## 2. Analyse des cibles (y)

In [ ]:
import matplotlib.pyplot as plt

y_preview = df[TARGETS]

# ── Stats de base ─────────────────────────────────────────────────────────────
print('Stats des cibles :')
print(y_preview.describe().round(1).T[['mean', 'std', 'min', '50%', 'max']])

# ── % de zéros ───────────────────────────────────────────────────────────────
print('\n% de zéros par cible :')
for col in TARGETS:
    pct = (y_preview[col] == 0).mean() * 100
    print(f'  {col.split(".")[1]:<15} {pct:.1f}%')

# ── Corrélation entre cibles ─────────────────────────────────────────────────
print('\nCorrélation entre cibles :')
print(y_preview.corr().round(2))

# ── Distributions ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(TARGETS), figsize=(18, 4))
for ax, col in zip(axes, TARGETS):
    data = y_preview[col]
    data[data > 0].plot(kind='hist', bins=50, ax=ax, color='steelblue', edgecolor='white', linewidth=0.3)
    ax.set_title(col.split('.')[1], fontsize=9)
    ax.set_xlabel('kWh / kg CO2e')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
plt.suptitle('Distribution des cibles (valeurs > 0)', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Gestion des NaN résiduels
- Colonnes > 30 % NaN → supprimées
- NaN restants (numériques) → médiane
- Lignes encore NaN → supprimées

In [ ]:
nan_pct = df.isna().mean()

drop_nan_cols = nan_pct[nan_pct > 0.30].index.tolist()
df.drop(columns=drop_nan_cols, inplace=True)
print(f'Colonnes supprimées (>30% NaN) : {len(drop_nan_cols)}')

nan_pct  = df.isna().mean()
num_cols = df.select_dtypes(include=[np.number]).columns
medians  = df[num_cols].median()
fill_cols = nan_pct[nan_pct > 0].index.intersection(num_cols)
df[fill_cols] = df[fill_cols].fillna(medians[fill_cols])
print(f'Colonnes imputées (médiane)    : {len(fill_cols)}')

rows_before = len(df)
df.dropna(inplace=True)
print(f'Lignes supprimées (NaN restants): {rows_before - len(df)}')

## 4. Séparation X / y
Toutes les colonnes qui ne sont ni des identifiants ni des cibles deviennent des features,
y compris les colonnes OHE qui n'ont plus le préfixe `in.` (climate_*, bldg_type_*, wall_type_*, hvac_*, etc.)

In [ ]:
# Colonnes à exclure : identifiants + cibles
IDENTIFIERS = ['bldg_id', 'upgrade', 'weight']
EXCLUDE = set(IDENTIFIERS) | set(TARGETS)

# Toutes les autres colonnes → features (in.* ET colonnes OHE sans préfixe)
feature_cols = [c for c in df.columns if c not in EXCLUDE]

X = df[feature_cols]
y = df[TARGETS]

# Vérification
in_cols    = [c for c in feature_cols if c.startswith('in.')]
other_cols = [c for c in feature_cols if not c.startswith('in.')]
print(f'X total     : {X.shape}')
print(f'  in.*      : {len(in_cols)} colonnes')
print(f'  OHE/autres: {len(other_cols)} colonnes -> {other_cols[:8]}...')
print(f'y           : {y.shape}')

## 5. Vérification de X avant sauvegarde

In [ ]:
print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')

print('\nTypes dans X :')
print(X.dtypes.value_counts())

non_numeriques = X.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeriques:
    print(f'\nColonnes non numériques ({len(non_numeriques)}) :')
    for c in non_numeriques:
        print(f'  {c} — {X[c].dtype}')
else:
    print('\nToutes les colonnes sont numériques.')

nan_cols = {c: round(X[c].isna().mean() * 100, 1) for c in X.columns if X[c].isna().any()}
print(f'\nColonnes avec NaN : {len(nan_cols)}')
for c, pct in nan_cols.items():
    print(f'  {c} — {pct}%')

## 5. Sauvegarde → `X.parquet` + `y.parquet`

In [ ]:
X.to_parquet(DATA_PROCESSED / 'X.parquet', index=False)
y.to_parquet(DATA_PROCESSED / 'y.parquet', index=False)

print(f'X sauvegardé : {DATA_PROCESSED}/X.parquet  {X.shape}')
print(f'y sauvegardé : {DATA_PROCESSED}/y.parquet  {y.shape}')